# 12_Lasso Regression

# Lasso Regression

Ridge Regression은 큰 Weight에 Penalty를 부여하여 Overfitting을 줄였다.

그러나 Ridge Regression에서는 일반적으로 Weight가 작아질 뿐 0이 되지는 않는다.

이번에는 다음 Problem을 생각해 보자.

> 중요하지 않은 Feature의 Weight를 0으로 만들 수 있을까?

Lasso Regression은 일부 Weight를 0으로 만들어 Feature Selection의 효과를 얻을 수 있는 Regression 방법이다.

## Ridge and Lasso

Ridge Regression:

$$
\min_w
\left[
\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
+
\alpha\sum_{j=1}^{p}w_j^2
\right]
$$

Lasso Regression:

$$
\min_w
\left[
\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
+
\alpha\sum_{j=1}^{p}|w_j|
\right]
$$

두 방법 모두 Weight에 Penalty를 부여한다.

Ridge는 Weight의 제곱을 사용하고 Lasso는 Weight의 Absolute Value를 사용한다.

## L1 Regularization

Lasso에서 사용하는

$$
\alpha\sum_{j=1}^{p}|w_j|
$$

를 L1 Regularization이라고 한다.

Lasso의 중요한 특징은 일부 Weight를 정확히 0으로 만들 수 있다는 것이다.

$$
w_j=0
$$

이면 해당 Feature는 Prediction에 사용되지 않는다.

따라서 Lasso는 Regression과 Feature Selection을 동시에 수행할 수 있다.

In [1]:
from google.colab import drive

# Google Drive를 Colab에 Mount
drive.mount('/content/drive')

# 강의용 Data가 저장된 Folder

data_path = "/content/drive/My Drive/Colab Notebooks/0_ML/Input/"

Mounted at /content/drive


In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

# Data
df = pd.read_csv(data_path+"lasso_data.csv")

X = df.drop(columns=["y"])
y = df["y"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

display(df.head())

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,y
0,-1.254599,1.420316,-3.968761,-3.310649,2.072386,-3.148671,2.582632,-3.329581,-3.081327,4.565014,-27.419411
1,4.507143,-4.158600,4.025529,-2.214097,-3.474610,0.419009,-4.754131,-3.323808,-1.766284,2.375084,17.687997
2,2.319939,-3.383713,0.052524,-3.229895,0.762884,3.729458,-4.778764,-4.633286,-2.733436,-1.467486,50.652232
3,0.986585,3.985542,3.264575,-4.112975,1.067150,2.322249,-1.763898,2.364020,-1.450037,-2.034644,15.579462
4,-3.439814,1.064291,-1.799504,-3.793641,-0.758693,3.065611,-0.113568,1.638045,-4.305762,-1.502968,6.332956


## Linear Regression

먼저 모든 Features를 이용하여 Linear Regression을 수행한다.

각 Feature의 Weight를 확인해 보자.

In [3]:
linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

linear_weights = pd.DataFrame({
    "Feature": X.columns,
    "Weight": linear_model.coef_
})

display(linear_weights)

print(
    "Test MSE:",
    mean_squared_error(
        y_test,
        linear_model.predict(X_test)
    )
)

,Feature,Weight
0,x1,7.962009
1,x2,0.176238
2,x3,-4.904479
3,x4,0.084714
4,x5,-0.134494
5,x6,9.918114
6,x7,0.063077
7,x8,-0.136198
8,x9,2.980679
9,x10,0.286603


Test MSE: 8.381798740368675


## Apply Lasso Regression

같은 Data에 Lasso Regression을 적용한다.

Lasso에서는 $\alpha$가 Regularization의 강도를 결정한다.

$\alpha$가 증가하면 Weight에 대한 Penalty가 강해진다.

In [8]:
lasso_model = Lasso(
    alpha=0.1
)

lasso_model.fit(
    X_train,
    y_train
)

lasso_weights = pd.DataFrame({
    "Feature": X.columns,
    "Weight": lasso_model.coef_
})

display(lasso_weights)

print(
    "Test MSE:",
    mean_squared_error(
        y_test,
        lasso_model.predict(X_test)
    )
)

,Feature,Weight
0,x1,7.951450
1,x2,0.168112
2,x3,-4.893215
3,x4,0.070567
4,x5,-0.117942
5,x6,9.910826
6,x7,0.049104
7,x8,-0.125790
8,x9,2.967399
9,x10,0.275753


Test MSE: 8.578149567003223


## Feature Selection

Lasso의 Weight를 관찰해 보자.

Weight가 0인 Feature:

$$
w_j=0
$$

해당 Feature는 Prediction에 영향을 주지 않는다.

Weight가 0이 아닌 Feature:

Model이 Prediction에 사용하는 Feature이다.

따라서 Lasso의 Coefficients를 관찰하면 Model이 어떤 Features를 선택했는지 확인할 수 있다.

In [9]:
# 선택된 Features
selected_features = lasso_weights[
    lasso_weights["Weight"] != 0
]

print("Selected Features")
display(selected_features)

print(
    "Number of Selected Features:",
    len(selected_features)
)

Selected Features


,Feature,Weight
0,x1,7.951450
1,x2,0.168112
2,x3,-4.893215
3,x4,0.070567
4,x5,-0.117942
5,x6,9.910826
6,x7,0.049104
7,x8,-0.125790
8,x9,2.967399
9,x10,0.275753


Number of Selected Features: 10


## Effect of Alpha

$\alpha$가 증가하면 Regularization이 강해진다.

그 결과 더 많은 Weight가 0이 될 수 있다.

하지만 $\alpha$가 지나치게 크면 중요한 Feature까지 제거될 수 있다.

따라서 $\alpha$에 따라

- Model Performance
- Number of Selected Features

가 어떻게 변하는지 함께 확인할 필요가 있다.

In [10]:
alphas = [0.01, 0.1, 1, 10]

for alpha in alphas:

    model = Lasso(
        alpha=alpha
    )

    model.fit(
        X_train,
        y_train
    )

    test_mse = mean_squared_error(
        y_test,
        model.predict(X_test)
    )

    selected = sum(
        model.coef_ != 0
    )

    print(
        f"alpha={alpha}: "
        f"Test MSE={test_mse:.3f}, "
        f"Selected Features={selected}"
    )

alpha=0.01: Test MSE=8.399, Selected Features=10
alpha=0.1: Test MSE=8.578, Selected Features=10
alpha=1: Test MSE=10.713, Selected Features=7
alpha=10: Test MSE=71.368, Selected Features=4


## Ridge vs Lasso

Ridge와 Lasso는 모두 Regularization을 이용한다.

Ridge:

$$
\alpha\sum w_j^2
$$

- L2 Regularization
- Weight를 작게 만든다.
- 일반적으로 Features를 유지한다.

Lasso:

$$
\alpha\sum |w_j|
$$

- L1 Regularization
- 일부 Weight를 0으로 만들 수 있다.
- Feature Selection 효과가 있다.

따라서 두 방법의 가장 중요한 차이는 Weight가 0이 될 수 있는가에 있다.

In [11]:
ridge_model = Ridge(alpha=1)
lasso_model = Lasso(alpha=1)

ridge_model.fit(X_train, y_train)
lasso_model.fit(X_train, y_train)

comparison = pd.DataFrame({
    "Feature": X.columns,
    "Ridge": ridge_model.coef_,
    "Lasso": lasso_model.coef_
})

display(comparison)

,Feature,Ridge,Lasso
0,x1,7.957174,7.868646
1,x2,0.176814,0.084264
2,x3,-4.900310,-4.785909
3,x4,0.086623,0.000000
4,x5,-0.134772,-0.000000
5,x6,9.910223,9.830354
6,x7,0.062775,0.000000
7,x8,-0.138789,-0.038262
8,x9,2.976738,2.847627
9,x10,0.286054,0.165331


## Summary

Ridge와 Lasso는 모두 Regularization을 이용하여 Weight를 제한한다.

Ridge Regression은 L2 Regularization을 사용한다.

Lasso Regression은 L1 Regularization을 사용한다.

Lasso에서는 일부 Weight가 정확히 0이 될 수 있다.

따라서 Lasso는

> Regression + Feature Selection

으로 이해할 수 있다.

$\alpha$가 증가하면 Regularization이 강해지고 더 많은 Features가 제거될 수 있다.

## Assignment - 문제 12

`Input/lasso_data2.csv`를 이용하여 Lasso Regression을 수행한다.

다음 조건을 사용한다.

- `test_size=0.2`
- `random_state=42`
- $\alpha = 0.01,\;0.1,\;1$

다음 질문에 답하시오.

- $\alpha=0.01$일 때 Weight가 0이 아닌 Feature의 수를 적으시오.
- $\alpha=0.1$일 때 Weight가 0이 아닌 Feature의 수를 적으시오.
- $\alpha=1$일 때 Weight가 0이 아닌 Feature의 수를 적으시오.
- $\alpha=0.1$일 때 Weight가 0인 Feature Name을 모두 적으시오.
- 세 Lasso Models 중 Test MSE가 가장 작은 $\alpha$와 그 Test MSE를 소수점 셋째 자리까지 적으시오.

답만 제출하시오.